#### 데이터 불러오기

In [4]:
import pandas as pd
from pathlib import Path
from collections import Counter
import unicodedata
import os
import re
import zlib
import fitz
import olefile

In [5]:
import pandas as pd
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 50)

In [7]:
df = pd.read_csv("/home/shared/data_list.csv", encoding="utf-8")
df.head(3)

,공고 번호,공고 차수,사업명,사업 금액,발주 기관,공개 일자,입찰 참여 시작일,입찰 참여 마감일,사업 요약,파일형식,파일명,텍스트
0,20241001798,0.0,한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화,130000000.0,한영대학,2024-10-04 13:51:23,NaN,2024-10-15 17:00:00,- 한영대학교 특성화 맞춤형 교육환경 구축을 위해 트랙운영 학사정보시스템을 고도화한...,hwp,한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp,\n \n2024년 특성화 맞춤형 교육환경 구축 – 트랙운영 학사정보시스템 ...
1,20241002912,0.0,2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선,129300000.0,한국연구재단,2024-10-04 15:01:52,2024-10-14 10:00:00,2024-10-16 14:00:00,- 사업 개요: 2024년 대학 산학협력활동 실태조사 시스템(UICC) 기능개선\n...,hwp,한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp,\r\n \r\n \r\n \r\n제 안 요 청 서\r\n[ 2024년 대학 ...
2,20240827859,0.0,EIP3.0 고압가스 안전관리 시스템 구축 용역,40000000.0,한국생산기술연구원,2024-08-28 11:31:02,2024-08-29 09:00:00,2024-09-09 10:00:00,- 사업 개요: EIP3.0 고압가스 안전관리 시스템 구축 용역\n- 추진배경: 안...,hwp,한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp,\r\n \r\nEIP3.0 고압가스 안전관리\r\n시스템 구축 용역\...


In [8]:
df.columns

Index(['공고 번호', '공고 차수', '사업명', '사업 금액', '발주 기관', '공개 일자', '입찰 참여 시작일',
       '입찰 참여 마감일', '사업 요약', '파일형식', '파일명', '텍스트'],
      dtype='str')

In [9]:
df.shape

(100, 12)

In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   공고 번호      82 non-null     str    
 1   공고 차수      82 non-null     float64
 2   사업명        100 non-null    str    
 3   사업 금액      99 non-null     float64
 4   발주 기관      100 non-null    str    
 5   공개 일자      100 non-null    str    
 6   입찰 참여 시작일  74 non-null     str    
 7   입찰 참여 마감일  92 non-null     str    
 8   사업 요약      100 non-null    str    
 9   파일형식       100 non-null    str    
 10  파일명        100 non-null    str    
 11  텍스트        100 non-null    str    
dtypes: float64(2), str(10)
memory usage: 853.9 KB


In [11]:
from pathlib import Path
from collections import Counter

folder = Path("/home/shared/files")

ext_counter = Counter()

for f in folder.rglob("*"):
    if f.is_file():
        ext = f.suffix.lower() if f.suffix else "(no_extension)"
        ext_counter[ext] += 1

print("확장자별 개수")
for ext, count in sorted(ext_counter.items()):
    print(f"{ext}: {count}")

확장자별 개수
.docx: 1
.hwp: 95
.pdf: 5


#### 문서이름 비교

In [12]:
import pandas as pd
import unicodedata

def normalize_name(x):
    x = str(x).strip()
    x = unicodedata.normalize("NFC", x)   # 한글 정규화
    return x

# 메타데이터 파일명
meta_names = (
    df["파일명"]
    .dropna()
    .astype(str)
    .map(normalize_name)
)

# files 폴더 파일명
files_names = pd.Series(
    [normalize_name(f.name) for f in folder.rglob("*") if f.is_file()]
)

# set 비교
only_in_meta = sorted(set(meta_names) - set(files_names))
only_in_files = sorted(set(files_names) - set(meta_names))

print("메타데이터에만 있는 파일 수:", len(only_in_meta))
print("files 폴더에만 있는 파일 수:", len(only_in_files))

print("\n[메타데이터에만 있는 파일명]")
for x in only_in_meta:
    print(x)

print("\n[files 폴더에만 있는 파일명]")
for x in only_in_files:
    print(x)

메타데이터에만 있는 파일 수: 1
files 폴더에만 있는 파일 수: 2

[메타데이터에만 있는 파일명]
한국농어촌공사_아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아.hwp

[files 폴더에만 있는 파일명]
고려대학교_차세대 포털·학사 정보시스템 구축사업.docx
한국농어촌공사_아세안+3+식량안보정보시스템(AFSIS)+3단계+협력(캄보디아.hwp.pdf


In [ ]:
# 고려대학교_차세대 포털·학사 정보시스템 구축사업: .docx .pdf 중복
# 메타데이터 ['파일명']이랑 files폴더내 문서들이랑 동일

#### 텍스트내용 <=500

In [14]:
short_rows2 = df.loc[df["텍스트"].fillna("").str.len() <= 500, ["파일명", "텍스트"]].copy()
short_rows2["텍스트길이"] = short_rows2["텍스트"].fillna("").str.len()

print(short_rows2[["파일명", "텍스트길이"]])

                                             파일명  텍스트길이
2       한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp    234
8      재단법인스포츠윤리센터_스포츠윤리센터 LMS(학습지원시스템) 기능개선.hwp    298
12    서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf    220
17  2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp    186
18   한국발명진흥회 입찰공고_2024년 건설기술에 관한 특허·실용신안 활용실.hwp     89
20        전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp    130
49     국민연금공단_사업장 사회보험료 지원 고시 개정에 따른 정보시스템 보.hwp    401


#### pdf 텍스트복구

In [15]:
import os
import re
import zlib
import fitz
import olefile
import pandas as pd

# 실제 파일 폴더 경로에 맞게 수정
FILES_DIR = "/home/shared/files"  # "/home/shared/files"
file_name = "서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf"
pdf_file = os.path.join(FILES_DIR, file_name)

# PDF 추출 + 정제 함수
def extract_and_clean_pdf_text(file_path):
    try:
        doc = fitz.open(file_path)
        full_text = ""
        for page in doc:
            full_text += page.get_text()

        print(f"[정제 전] 텍스트 길이: {len(full_text)}")

        # 1차 정제: 불필요한 특수기호 제거
        cleaned_text = re.sub(
            r'[^가-힣a-zA-Z0-9\s\.\(\)\[\]\/\,\%\:\-\·\?\!\@]',
            ' ',
            full_text
        )

        # 2차 정제: 목차 점선/연속 점 제거
        cleaned_text = re.sub(r'[\.·]{2,}', ' ', cleaned_text)

        # 3차 정제: 다중 공백 정리
        cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()

        print(f"[정제 후] 텍스트 길이: {len(cleaned_text)}")
        print("-" * 60)

        return cleaned_text

    except Exception as e:
        return f"PDF 추출 실패: {str(e)}"

cleaned_text = extract_and_clean_pdf_text(pdf_file)
print(f"새로 추출한 텍스트 길이: {len(cleaned_text)}")
print(cleaned_text[:500])

[정제 전] 텍스트 길이: 116224
[정제 후] 텍스트 길이: 104123
------------------------------------------------------------
새로 추출한 텍스트 길이: 104123
[사전공개용] 제 안 요 청 서 본 제안요청서는 입찰참여의 균등한 기회 제공을 위해 규격을 공개하기 위한 자료로써 실제 입찰공고 시 사업금액, 과업내용, 평가항목, 제출서류 등은 변경될 수 있으니 반드시 확인하시기 바랍니다. 2023. 06. 담당 성명 소 속 전화번호 e-mail 이석준 서울시립대학교 (입학처) 02-6490-6176 lsjptrs@uos.ac.kr 사 업 명 학업성취도 다차원 종단분석 통합시스템 1차 고도화 주관기관 서 울 시 립 대 학 교 입 학 처 목 차 . 사업안내 1. 사업개요 01 2. 추진배경 및 필요성 01 3. 사업근거 및 3개년 추진계획 01 4. 사업범위 02 5. 기대효과 03 . 대상업무 현황 1. 기존 연계 현황 03 2. 개발 완료 내역 04 3. 추가 연계 필요 현황 06 4. 시스템 현황 06 . 사업추진 방안 1. 추진체계 07 2. 추진일정 08 3. 추진방안 09 . 제안요청내용 1. 시스템 개발 범위 09 2. 제안요청 내용 1


In [17]:
# 복구한 거 덮어쓰기
df.loc[df["파일명"] == file_name, "텍스트"] = cleaned_text
df.loc[df["파일명"] == file_name, "텍스트길이"] = len(cleaned_text)

In [18]:
row = df.loc[df["파일명"] == "서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf", ["파일명", "텍스트"]]
text = row["텍스트"].fillna("").iloc[0]

print("텍스트길이:", len(text))
# print(text[:500])

텍스트길이: 104123


#### hwp 텍스트복구

In [19]:
FILES_DIR = "/home/shared/files"

def get_hwp_text_final_exorcism(file_path):
    def is_clean_hangul(char):
        if not '가' <= char <= '힣':
            return True
        try:
            code = char.encode('cp949')
            return 0xB0 <= code[0] <= 0xC8
        except:
            return False

    try:
        f = olefile.OleFileIO(file_path)
        dirs = f.listdir()
        bodytext_sections = [d for d in dirs if 'BodyText' in d]

        raw_text = ""
        for section in bodytext_sections:
            data = f.openstream(section).read()
            decompressed = zlib.decompress(data, -15)
            raw_text += decompressed.decode('utf-16', errors='ignore')

        # 이미지 정보 및 불필요한 메타데이터 제거
        text = re.sub(r'원본 그림의 이름:.*?pixel', ' ', raw_text, flags=re.DOTALL)
        text = re.sub(r'가로 \d+pixel, 세로 \d+pixel', ' ', text)
        text = re.sub(r'[^가-힣a-zA-Z0-9\s\.\(\)\[\]\/\,\%\:\-\·\?\!]', ' ', text)

        tokens = text.split()
        clean_tokens = []

        for t in tokens:
            if all(is_clean_hangul(c) for c in t):
                if len(t) == 1 and t not in '이가을를에와과도한1234567890o-·':
                    continue
                if re.search(r'[a-zA-Z][가-힣]', t) or re.search(r'[가-힣][a-zA-Z]', t):
                    continue
                clean_tokens.append(t)

        return " ".join(clean_tokens)

    except Exception as e:
        print(f"추출 실패: {file_path} / {e}")
        return None


remaining_files = [
    "한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp",
    "재단법인스포츠윤리센터_스포츠윤리센터 LMS(학습지원시스템) 기능개선.hwp",
    "2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp",
    "전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp",
    "한국발명진흥회 입찰공고_2024년 건설기술에 관한 특허·실용신안 활용실.hwp",
    "국민연금공단_사업장 사회보험료 지원 고시 개정에 따른 정보시스템 보.hwp"
]

print("HWP 복구를 시작합니다...")
print("-" * 40)

for file_name in remaining_files:
    file_path = os.path.join(FILES_DIR, file_name)
    idx_list = df[df["파일명"] == file_name].index

    if not idx_list.empty:
        idx = idx_list[0]
        print(f"처리 중: {file_name}")

        restored_text = get_hwp_text_final_exorcism(file_path)

        if restored_text and len(restored_text) > 100:
            df.loc[idx, "텍스트"] = restored_text
            df.loc[idx, "텍스트길이"] = len(restored_text)
            print(f"복구 성공! (최종 길이: {len(restored_text):,}자)")
        else:
            print("텍스트가 너무 짧거나 추출 실패")
    else:
        print(f"df에서 파일명을 찾지 못함: {file_name}")

print("-" * 40)
print("HWP 복구 완료")

display(df[df["파일명"].isin(remaining_files)][["파일명", "텍스트길이"]])

HWP 복구를 시작합니다...
----------------------------------------
처리 중: 한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp
복구 성공! (최종 길이: 54,439자)
처리 중: 재단법인스포츠윤리센터_스포츠윤리센터 LMS(학습지원시스템) 기능개선.hwp
복구 성공! (최종 길이: 35,327자)
처리 중: 2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp
복구 성공! (최종 길이: 26,973자)
처리 중: 전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp
복구 성공! (최종 길이: 34,048자)
처리 중: 한국발명진흥회 입찰공고_2024년 건설기술에 관한 특허·실용신안 활용실.hwp
복구 성공! (최종 길이: 27,835자)
처리 중: 국민연금공단_사업장 사회보험료 지원 고시 개정에 따른 정보시스템 보.hwp
복구 성공! (최종 길이: 31,210자)
----------------------------------------
HWP 복구 완료


,파일명,텍스트길이
2,한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp,54439.0
8,재단법인스포츠윤리센터_스포츠윤리센터 LMS(학습지원시스템) 기능개선.hwp,35327.0
17,2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp,26973.0
18,한국발명진흥회 입찰공고_2024년 건설기술에 관한 특허·실용신안 활용실.hwp,27835.0
20,전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp,34048.0
49,국민연금공단_사업장 사회보험료 지원 고시 개정에 따른 정보시스템 보.hwp,31210.0


In [20]:
# 복구최종체크
check_files = [
    "서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf",
    "한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp",
    "재단법인스포츠윤리센터_스포츠윤리센터 LMS(학습지원시스템) 기능개선.hwp",
    "2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp",
    "전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp",
    "한국발명진흥회 입찰공고_2024년 건설기술에 관한 특허·실용신안 활용실.hwp",
    "국민연금공단_사업장 사회보험료 지원 고시 개정에 따른 정보시스템 보.hwp"
]

for file_name in check_files:
    row = df.loc[df["파일명"] == file_name, ["파일명", "텍스트"]]
    print("파일명:", file_name)

    if row.empty:
        print("df에서 해당 파일을 찾지 못했습니다.")
        continue

    text = row["텍스트"].fillna("").iloc[0]
    print("텍스트길이:", len(text))
    print()

파일명: 서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf
텍스트길이: 104123

파일명: 한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp
텍스트길이: 54439

파일명: 재단법인스포츠윤리센터_스포츠윤리센터 LMS(학습지원시스템) 기능개선.hwp
텍스트길이: 35327

파일명: 2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp
텍스트길이: 26973

파일명: 전북대학교_JST 공유대학(원) xAPI기반 LRS시스템 구축.hwp
텍스트길이: 34048

파일명: 한국발명진흥회 입찰공고_2024년 건설기술에 관한 특허·실용신안 활용실.hwp
텍스트길이: 27835

파일명: 국민연금공단_사업장 사회보험료 지원 고시 개정에 따른 정보시스템 보.hwp
텍스트길이: 31210



#### 사업요약 vs 텍스트

In [21]:
check_len = df[["파일명", "텍스트", "사업 요약"]].copy()

check_len["텍스트길이"] = check_len["텍스트"].fillna("").str.len()
check_len["사업요약길이"] = check_len["사업 요약"].fillna("").str.len()

long_summary = check_len[check_len["사업요약길이"] > check_len["텍스트길이"]]

print(long_summary[["파일명", "텍스트길이", "사업요약길이"]])

Empty DataFrame
Columns: [파일명, 텍스트길이, 사업요약길이]
Index: []


In [ ]:
# 사업 요약이 텍스트보다 긴 문서 없음

In [22]:
output_path = "/home/bidcoin/data_cleaning1.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"저장 완료: {output_path}")

저장 완료: /home/bidcoin/data_cleaning1.csv


#### 텍스트복구 불러오기

In [23]:
df2 = pd.read_csv("/home/bidcoin/data_cleaning1.csv", encoding="utf-8")
df2.shape

(100, 13)

In [24]:
df2.columns

Index(['공고 번호', '공고 차수', '사업명', '사업 금액', '발주 기관', '공개 일자', '입찰 참여 시작일',
       '입찰 참여 마감일', '사업 요약', '파일형식', '파일명', '텍스트', '텍스트길이'],
      dtype='str')

#### pdf/hwp 길이확인

In [25]:
pdf_text_len = df2.loc[df2['파일형식'].astype(str).str.lower() == 'pdf', '텍스트'].fillna('').str.len()

print("최대 길이:", pdf_text_len.max())
print("최소 길이:", pdf_text_len.min())

최대 길이: 104123
최소 길이: 808


In [26]:
pdf_text_len

7       2450
12    104123
39       808
50      2716
Name: 텍스트, dtype: int64

In [27]:
hwp_text_len = df2.loc[df2['파일형식'].astype(str).str.lower() == 'hwp', '텍스트'].fillna('').str.len()

print("최대 길이:", hwp_text_len.max())
print("최소 길이:", hwp_text_len.min())

최대 길이: 54439
최소 길이: 533


In [28]:
hwp_text_len.describe()

count       96.000000
mean      6110.968750
std       8629.790238
min        533.000000
25%       1889.000000
50%       3391.000000
75%       6234.750000
max      54439.000000
Name: 텍스트, dtype: float64

#### 입찰 참여 시작일

In [29]:
cols = ["파일명", "공개 일자", "입찰 참여 시작일", "입찰 참여 마감일"]

missing_start = df2.loc[
    df2["입찰 참여 시작일"].isna(),
    cols
].copy()

missing_start

,파일명,공개 일자,입찰 참여 시작일,입찰 참여 마감일
0,한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp,2024-10-04 13:51:23,NaN,2024-10-15 17:00:00
12,서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf,2023-06-20 00:00:00,NaN,NaN
13,경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp,2024-05-02 00:00:00,NaN,NaN
14,한국수자원공사_건설통합시스템(CMS) 고도화.hwp,2024-05-31 00:00:00,NaN,NaN
17,2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp,2024-09-10 16:31:49,NaN,2024-09-23 18:00:00
23,수협중앙회_강릉어선안전조업국 상황관제시스템 구축.hwp,2025-02-11 10:27:38,NaN,2025-03-10 11:00:00
24,한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp,2024-05-13 00:00:00,NaN,NaN
26,KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp,2024-10-24 00:00:00,NaN,NaN
31,인천광역시 동구_수도국산달동네박물관 전시해설 시스템 구축(협상에 .hwp,2025-01-24 19:56:15,NaN,2025-02-20 18:00:00
38,광주과학기술원_학사시스템 기능개선 사업.hwp,2024-12-09 08:52:59,NaN,2024-12-20 14:00:00


In [30]:
print(missing_start['파일명'])

0            한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp
12           서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf
13              경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp
14                         한국수자원공사_건설통합시스템(CMS) 고도화.hwp
17         2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp
23                       수협중앙회_강릉어선안전조업국 상황관제시스템 구축.hwp
24             한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp
26    KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp
31             인천광역시 동구_수도국산달동네박물관 전시해설 시스템 구축(협상에 .hwp
38                            광주과학기술원_학사시스템 기능개선 사업.hwp
41                            을지대학교_을지대학교 비교과시스템 개발.hwp
42      대전대학교_대전대학교 2024학년도 다층적 융합 학습경험 플랫폼(MILE) 전.hwp
46         BioIN_의료기기산업 종합정보시스템(정보관리기관) 기능개선 사업(2차).hwp
47                  한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp
58          수협중앙회_수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수립 입.hwp
59            대한상공회의소_기업 재생에너지 지원센터 홈페이지 개편 및 시스템 고.hwp
61          케빈랩 주식회사_평택시 강소형 스마트시티 AI 기반의 영상감시 시스템 .hwp
70          국가철도공단_철도인프라 디지털트윈 정보화전략계획(ISP) 수립 용

In [35]:
missing_start_list = missing_start["파일명"].tolist()
len(missing_start_list)

26

In [31]:
# 입찰 시작일 not null인거
cols = ["파일명", "공개 일자", "입찰 참여 시작일", "입찰 참여 마감일"]

not_null_start = df2.loc[
    df2["입찰 참여 시작일"].notna() &
    (df2["입찰 참여 시작일"].astype(str).str.strip() != ""),
    cols
].head(10)

display(not_null_start)

,파일명,공개 일자,입찰 참여 시작일,입찰 참여 마감일
1,한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp,2024-10-04 15:01:52,2024-10-14 10:00:00,2024-10-16 14:00:00
2,한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp,2024-08-28 11:31:02,2024-08-29 09:00:00,2024-09-09 10:00:00
3,인천광역시_도시계획위원회 통합관리시스템 구축용역.hwp,2024-04-18 16:26:32,2024-05-02 10:00:00,2024-05-09 16:00:00
4,경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp,2024-04-18 16:33:28,2024-04-26 09:00:00,2024-04-30 17:00:00
5,한국전기안전공사_전기안전 관제시스템 보안 모듈 개발 용역.hwp,2025-01-08 09:45:32,2025-01-08 14:30:00,2025-01-08 15:30:00
6,재단법인충북연구원_GIS통계 기반 재난안전데이터 분석ㆍ관리 시스템 구.hwp,2025-01-08 16:05:19,2025-01-09 10:00:00,2025-02-03 16:00:00
7,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,2024-07-01 00:00:00,2024-07-05 11:00:00,2024-08-12 11:00:00
8,재단법인스포츠윤리센터_스포츠윤리센터 LMS(학습지원시스템) 기능개선.hwp,2024-08-22 11:11:14,2024-08-22 14:00:00,2024-09-02 14:00:00
9,국방과학연구소_대용량 자료전송시스템 고도화.hwp,2024-08-22 11:20:10,2024-08-22 11:30:00,2024-10-01 10:00:00
10,(사）한국대학스포츠협의회_KUSF 체육특기자 경기기록 관리시스템 개발.hwp,2024-08-16 08:52:03,2024-09-02 10:00:00,2024-09-06 10:00:00


In [ ]:
# 공개일자랑 참여시작일이랑 날짜 같은 경우도 있지만 차이 많이 나는 경우도 있음

In [32]:
# 공개일 참여시작일 차이
cols = ["파일명", "공개 일자", "입찰 참여 시작일"]

temp = df2[cols].copy()

# 날짜형 변환
temp["공개 일자_dt"] = pd.to_datetime(temp["공개 일자"], errors="coerce")
temp["입찰 참여 시작일_dt"] = pd.to_datetime(temp["입찰 참여 시작일"], errors="coerce")

# 날짜 차이 계산
temp["차이일수"] = (temp["입찰 참여 시작일_dt"] - temp["공개 일자_dt"]).dt.days

# 둘 다 날짜가 있는 것만, 큰 순서대로 top 10
top10_gap = (
    temp.dropna(subset=["공개 일자_dt", "입찰 참여 시작일_dt"])
        .sort_values("차이일수", ascending=False)
        .head(10)
)

display(top10_gap[["파일명", "공개 일자", "입찰 참여 시작일", "차이일수"]])

,파일명,공개 일자,입찰 참여 시작일,차이일수
94,한국산업단지공단_산단 안전정보시스템 1차 구축 용역.hwp,2024-04-04 15:09:35,2024-05-13 10:00:00,38.0
89,한국수출입은행_(긴급) 모잠비크 마푸토 지능형교통시스템(ITS) 구축사업.hwp,2024-10-04 09:34:21,2024-10-28 10:00:00,24.0
49,국민연금공단_사업장 사회보험료 지원 고시 개정에 따른 정보시스템 보.hwp,2024-06-05 00:00:00,2024-06-28 10:00:00,23.0
67,한국교육과정평가원_국가교육과정정보센터(NCIC) 시스템 운영 및 개선.hwp,2024-05-24 09:47:11,2024-06-12 10:00:00,19.0
10,(사）한국대학스포츠협의회_KUSF 체육특기자 경기기록 관리시스템 개발.hwp,2024-08-16 08:52:03,2024-09-02 10:00:00,17.0
27,한국수자원공사_수도사업장 통합 사고분석솔루션 시범구축 용역.hwp,2024-05-07 00:00:00,2024-05-24 09:00:00,17.0
39,서울특별시_2024년 지도정보 플랫폼 및 전문활용 연계 시스템 고도화 용.pdf,2024-04-02 15:49:39,2024-04-19 09:00:00,16.0
3,인천광역시_도시계획위원회 통합관리시스템 구축용역.hwp,2024-04-18 16:26:32,2024-05-02 10:00:00,13.0
91,한국보육진흥원_연차별 자율 품질관리 시스템 기능개선.hwp,2024-04-18 10:59:30,2024-04-30 10:00:00,11.0
33,울산광역시_2024년 버스정보시스템 확대 구축 및 기능개선 용역.hwp,2024-05-20 17:23:27,2024-05-31 09:00:00,10.0


In [33]:
# 시작일은 마감일에 비해 중요도가 낮으니 그냥 공개일자로 대체
mask = df2["입찰 참여 시작일"].isna() & df2["공개 일자"].notna()

df2.loc[mask, "입찰 참여 시작일"] = df2.loc[mask, "공개 일자"]

In [36]:
# 확인
df2.loc[
    df2["파일명"].isin(missing_start_list),
    ["파일명", "공개 일자", "입찰 참여 시작일"]
]

,파일명,공개 일자,입찰 참여 시작일
0,한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp,2024-10-04 13:51:23,2024-10-04 13:51:23
12,서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf,2023-06-20 00:00:00,2023-06-20 00:00:00
13,경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp,2024-05-02 00:00:00,2024-05-02 00:00:00
14,한국수자원공사_건설통합시스템(CMS) 고도화.hwp,2024-05-31 00:00:00,2024-05-31 00:00:00
17,2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp,2024-09-10 16:31:49,2024-09-10 16:31:49
23,수협중앙회_강릉어선안전조업국 상황관제시스템 구축.hwp,2025-02-11 10:27:38,2025-02-11 10:27:38
24,한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp,2024-05-13 00:00:00,2024-05-13 00:00:00
26,KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp,2024-10-24 00:00:00,2024-10-24 00:00:00
31,인천광역시 동구_수도국산달동네박물관 전시해설 시스템 구축(협상에 .hwp,2025-01-24 19:56:15,2025-01-24 19:56:15
38,광주과학기술원_학사시스템 기능개선 사업.hwp,2024-12-09 08:52:59,2024-12-09 08:52:59


#### 입찰 참여 마감일

In [37]:
cols2 = ["파일명", "공개 일자", "입찰 참여 시작일", "입찰 참여 마감일"]

missing_end = df2.loc[
    df2["입찰 참여 마감일"].isna(),
    cols2
].copy()

missing_end

,파일명,공개 일자,입찰 참여 시작일,입찰 참여 마감일
12,서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf,2023-06-20 00:00:00,2023-06-20 00:00:00,NaN
13,경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp,2024-05-02 00:00:00,2024-05-02 00:00:00,NaN
14,한국수자원공사_건설통합시스템(CMS) 고도화.hwp,2024-05-31 00:00:00,2024-05-31 00:00:00,NaN
24,한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp,2024-05-13 00:00:00,2024-05-13 00:00:00,NaN
26,KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp,2024-10-24 00:00:00,2024-10-24 00:00:00,NaN
46,BioIN_의료기기산업 종합정보시스템(정보관리기관) 기능개선 사업(2차).hwp,2024-09-05 00:00:00,2024-09-05 00:00:00,NaN
70,국가철도공단_철도인프라 디지털트윈 정보화전략계획(ISP) 수립 용역(변.hwp,2024-08-13 00:00:00,2024-08-13 00:00:00,NaN
88,세종테크노파크_세종테크노파크 인사정보 전산시스템 구축 용역 입찰공.hwp,2021-10-08 00:00:00,2021-10-08 00:00:00,NaN


In [38]:
print(missing_end['파일명'])

12           서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf
13              경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp
14                         한국수자원공사_건설통합시스템(CMS) 고도화.hwp
24             한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp
26    KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp
46         BioIN_의료기기산업 종합정보시스템(정보관리기관) 기능개선 사업(2차).hwp
70          국가철도공단_철도인프라 디지털트윈 정보화전략계획(ISP) 수립 용역(변.hwp
88             세종테크노파크_세종테크노파크 인사정보 전산시스템 구축 용역 입찰공.hwp
Name: 파일명, dtype: str


In [ ]:
'''제출,마감,기한,입찰,기간 ( 입찰공고문...이건 제안요청서..)
12 : ...
13 : 2024년 05월 14일 11:00AM 마감
14 : ...
24 : ...
26 : ...
46 : ...
70 : ...
88 : ...
''';

In [41]:
# 직접 입력
df2.loc[
    df2["파일명"] == "경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp",
    "입찰 참여 마감일"
] = "2024-05-14 11:00:00"

In [42]:
mask = df2["파일명"] == "경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp"

df2.loc[mask, ["파일명", "입찰 참여 마감일"]]

,파일명,입찰 참여 마감일
13,경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp,2024-05-14 11:00:00


In [ ]:
# 마감일은 RFP문서에 나와있지 않고 입찰공고문을 참고하라고 함
# 나머지는 결측채우기보다 널값으로 두는게 낫다는 판단

#### 사업 금액

In [43]:
display(df2.loc[df2["사업 금액"].isna(), ["파일명","사업 금액"]])

,파일명,사업 금액
59,대한상공회의소_기업 재생에너지 지원센터 홈페이지 개편 및 시스템 고.hwp,NaN


In [44]:
# 직접 입력
df2.loc[df2["사업 금액"].isna(), "사업 금액"] = 57000000
display(df2.loc[df2["파일명"] == "대한상공회의소_기업 재생에너지 지원센터 홈페이지 개편 및 시스템 고.hwp", ["파일명", "사업 금액"]])

,파일명,사업 금액
59,대한상공회의소_기업 재생에너지 지원센터 홈페이지 개편 및 시스템 고.hwp,57000000.0


In [48]:
df2["사업 금액"].isna().sum() # 널값없어짐

np.int64(0)

In [49]:
df2['사업 금액'].min(), df2['사업 금액'].max()  # 0인것 발견

(np.float64(0.0), np.float64(14107009000.0))

In [47]:
df2[['파일명', '사업 금액']].nsmallest(10, '사업 금액')

,파일명,사업 금액
12,서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf,0.0
13,경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp,0.0
16,한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp,0.0
34,한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총체 및 1차).hwp,0.0
41,을지대학교_을지대학교 비교과시스템 개발.hwp,0.0
71,한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp,0.0
83,사단법인 보험개발원_실손보험 청구 전산화 시스템 구축 사업.hwp,1.0
52,대검찰청_아태 사이버범죄 역량강화 허브(APC-HUB) 홈페이지 및 온라인 교.hwp,35750000.0
2,한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp,40000000.0
74,재단법인 광주연구원_광주정책연구아카이브(GPA) 시스템 개발.hwp,43000000.0


In [ ]:
# 0인거 채워넣어야 할듯

In [ ]:
''' 단위는 <원>
12 : 242,900,000원
13 : 1차년도(계약체결일 ~ 2025.04.30.) 200,000,000원, 2차년도(2025.05.01. ~ 2026.04.30.) 200,000,000원
16 : 470백만원
34 : 개발비 359백만원 + H/W 484백만원(VAT 포함) = 843백만원
41 : 비공개
71 : 비공개
83 : 비공개
52 : 35,750천원 = 35,750,000
'''

In [50]:
# 직접입력
df2.loc[df2["파일명"] == "서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf", "사업 금액"] = 242900000
df2.loc[df2["파일명"] == "경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp", "사업 금액"] = 200000000
df2.loc[df2["파일명"] == "한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp", "사업 금액"] = 470000000
df2.loc[df2["파일명"] == "한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총체 및 1차).hwp", "사업 금액"] = 843000000
df2.loc[df2["파일명"] == "사단법인 보험개발원_실손보험 청구 전산화 시스템 구축 사업.hwp", "사업 금액"] = 0

In [51]:
# 확인
display(df2.loc[df2["파일명"] == "서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf", ["파일명", "사업 금액"]])
display(df2.loc[df2["파일명"] == "경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp", ["파일명", "사업 금액"]])
display(df2.loc[df2["파일명"] == "한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp", ["파일명", "사업 금액"]])
display(df2.loc[df2["파일명"] == "한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총체 및 1차).hwp", ["파일명", "사업 금액"]])
display(df2.loc[df2["파일명"] == "사단법인 보험개발원_실손보험 청구 전산화 시스템 구축 사업.hwp", ["파일명", "사업 금액"]])

,파일명,사업 금액
12,서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf,242900000.0


,파일명,사업 금액
13,경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp,200000000.0


,파일명,사업 금액
16,한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp,470000000.0


,파일명,사업 금액
34,한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총체 및 1차).hwp,843000000.0


,파일명,사업 금액
83,사단법인 보험개발원_실손보험 청구 전산화 시스템 구축 사업.hwp,0.0


#### 공고 번호

In [52]:
df2[['공고 번호']].value_counts()

공고 번호        
20241001798      1
20241002912      1
20240827859      1
20240430918      1
20240430896      1
R25BK00559883    1
R25BK00564730    1
20240821865      1
20240821893      1
20240812818      1
20240815487      1
20240910050      1
20240903676      1
20240903688      1
20240904268      1
R25BK00632049    1
R25BK00632248    1
R25BK00601569    1
R25BK00603644    1
R25BK00603533    1
R25BK00604826    1
R25BK00605617    1
20240523741      1
20240524568      1
20240413838      1
20240414353      1
20240345257      1
20241213403      1
20240404154      1
20241138828      1
20241138864      1
20241139040      1
20240723270      1
20240723668      1
20241130016      1
20241120435      1
20241118572      1
20240531013      1
20240535775      1
20240539319      1
20240539643      1
20240611568      1
20240611774      1
20240605067      1
20240605351      1
20240605366      1
20241211469      1
20241217596      1
20241218257      1
20240541684      1
20240541779      1
20241207733      

In [53]:
df2.loc[df2['공고 번호'].astype(str).str.startswith('R'), ['파일명','발주 기관']]  # 9 

,파일명,발주 기관
5,한국전기안전공사_전기안전 관제시스템 보안 모듈 개발 용역.hwp,한국전기안전공사
6,재단법인충북연구원_GIS통계 기반 재난안전데이터 분석ㆍ관리 시스템 구.hwp,재단법인충북연구원
22,한국사회보장정보원_라오스 보건의료정보화 협력을 위한 사전타당성 조.hwp,한국사회보장정보원
23,수협중앙회_강릉어선안전조업국 상황관제시스템 구축.hwp,수협중앙회
25,한국농어촌공사_아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아.hwp,한국농어촌공사
28,국방과학연구소_기록관리시스템 통합 활용 및 보안 환경 구축.hwp,국방과학연구소
29,인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역.hwp,인천광역시
30,"대한장애인체육회_2025년 전국장애인체육대회 전산 및 시스템, 홈페이지 .hwp",대한장애인체육회
31,인천광역시 동구_수도국산달동네박물관 전시해설 시스템 구축(협상에 .hwp,인천광역시 동구


In [58]:
# R로 시작하는 공고번호의 발주기관 리스트
agency_list = df2.loc[
    df2['공고 번호'].astype(str).str.startswith('R'),
    '발주 기관'
].drop_duplicates().tolist()
len(agency_list)

9

In [55]:
df2.loc[df2['공고 번호'].astype(str).str.startswith('2'), ['파일명','발주 기관']]  #73

,파일명,발주 기관
0,한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp,한영대학
1,한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp,한국연구재단
2,한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp,한국생산기술연구원
3,인천광역시_도시계획위원회 통합관리시스템 구축용역.hwp,인천광역시
4,경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp,경상북도 봉화군
8,재단법인스포츠윤리센터_스포츠윤리센터 LMS(학습지원시스템) 기능개선.hwp,재단법인스포츠윤리센터
9,국방과학연구소_대용량 자료전송시스템 고도화.hwp,국방과학연구소
10,(사）한국대학스포츠협의회_KUSF 체육특기자 경기기록 관리시스템 개발.hwp,(사）한국대학스포츠협의회
11,한국사학진흥재단_대학재정정보시스템(기본재산 및 기채 사후관리) 고.hwp,한국사학진흥재단
17,2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp,2025 구미 아시아육상경기선수권대회 조직위원회


In [59]:
# 2로 시작하는 공고번호의 발주기관 리스트
agency_list2 = df2.loc[
    df2['공고 번호'].astype(str).str.startswith('2'),
    '발주 기관'
].dropna().drop_duplicates().tolist()
len(agency_list2)

69

In [ ]:
# 공고번호가 발주기관에 따라 다르게 적용되는지 확인하기 위함

In [60]:
overlap = set(agency_list) & set(agency_list2)
print(overlap)
print(len(overlap))

{'인천광역시', '국방과학연구소', '한국농어촌공사', '수협중앙회'}
4


In [ ]:
# 겹치는게 있는 것을 보니 발주기관에 따라 공고번호 패턴이 다른건 아닌듯
# 공고번호를 가짜임이 드러나는 값으로 채우고자함

In [61]:
mask = df2['공고 번호'].isna()
df2.loc[mask, '공고 번호'] = 'MISSING_' + df2.loc[mask].index.astype(str)

In [62]:
df2.loc[df2['공고 번호'].astype(str).str.startswith('MISSING_'), ['파일명', '공고 번호']]

,파일명,공고 번호
7,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,MISSING_7
12,서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf,MISSING_12
13,경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp,MISSING_13
14,한국수자원공사_건설통합시스템(CMS) 고도화.hwp,MISSING_14
15,국가과학기술지식정보서비스_통합정보시스템 고도화 용역.hwp,MISSING_15
16,한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp,MISSING_16
18,한국발명진흥회 입찰공고_2024년 건설기술에 관한 특허·실용신안 활용실.hwp,MISSING_18
24,한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp,MISSING_24
26,KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp,MISSING_26
27,한국수자원공사_수도사업장 통합 사고분석솔루션 시범구축 용역.hwp,MISSING_27


#### 공고 차수

In [63]:
df2[['공고 차수']].value_counts() # 82

공고 차수
0.0      76
1.0       4
2.0       2
Name: count, dtype: int64

In [64]:
df2.loc[df2['공고 차수'].isna(), ['파일명']]  # 18

,파일명
7,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf
12,서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf
13,경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp
14,한국수자원공사_건설통합시스템(CMS) 고도화.hwp
15,국가과학기술지식정보서비스_통합정보시스템 고도화 용역.hwp
16,한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp
18,한국발명진흥회 입찰공고_2024년 건설기술에 관한 특허·실용신안 활용실.hwp
24,한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp
26,KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp
27,한국수자원공사_수도사업장 통합 사고분석솔루션 시범구축 용역.hwp


In [ ]:
# 공고차수도 마감일과 같은 맥락으로 그냥 널값으로 두는게 낫다는 판단

In [65]:
# 저장
output_path = "/home/bidcoin/data_cleaning2.csv"
df2.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"저장 완료: {output_path}")

저장 완료: /home/bidcoin/data_cleaning2.csv
